In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [3]:
df = pd.read_csv('../data/cleaned/India.csv')

In [4]:
df.shape

(73632, 8)

In [ ]:
df[df['Category'] == 'Child Vaccinations and Vitamin A Supplementation']['Indicator'].unique()

In [6]:
vacc_df = df[df['Indicator'] == "Children age 12-23 months fully vaccinated based on information from either vaccination card or mother's recall (%)"] 

filter children age 12-23 months... as the primary indicator; assign it to vacc_df

In [ ]:
vacc_df.shape

In [ ]:
sns.histplot(data=vacc_df, x = "NFHS_5")
plt.title("Distribution of full vaccination rates across Indian districts(NFHS-5)")
plt.xlabel("Full vaccination rate (%)")
plt.ylabel("Number of districts")
plt.savefig("Distribution of full vaccination rates across Indian districts(NFHS-5).png")
plt.show()

Most indian districts have full vaccination rates between 60-95%, with the peak around 75%. But, approximately 14 districts show near-zero vaccination rates, suggesting serious gaps or health failures in those areas; need to investigate

In [ ]:
vacc_df[vacc_df["NFHS_5"] < 5]

Filtered 9 states + 13 districts (except for J&K) with NFHS_5 = 0; NFHS_4 data is consistent; seems to be a data collection issue 

In [10]:
vacc_clean = vacc_df[vacc_df["NFHS_5"] > 0]

In [11]:
zscores = stats.zscore(vacc_clean['NFHS_5'])

In [ ]:
vacc_clean[abs(zscores) > 3]

Three Northeast districts — Udalguri (Assam), Ukhrul (Manipur), and Tuensang (Nagaland) — are statistical outliers with vaccination rates below 40%, compared to the national mean of ~76%. All three declined between NFHS-4 and NFHS-5, bucking the national improvement trend.

In [13]:
northeast_vacc = vacc_clean[vacc_clean['State'].isin(['Assam', 'Manipur', 'Nagaland', 'Meghalaya', 'Mizoram', 'Tripura', 'Arunachal Pradesh', 'Sikkim'])]

In [ ]:
northeast_vacc.shape

In [15]:
northeast_state_avg = northeast_vacc.groupby('State')[['NFHS_5', 'NFHS_4']].mean()

In [16]:
northeast_state_avg = northeast_state_avg.reset_index()

In [ ]:
northeast_state_avg.columns

In [18]:
northeast_vacc_melted = pd.melt(northeast_state_avg, id_vars = ['State'], value_vars = ['NFHS_5', 'NFHS_4'])

In [ ]:
plt.figure(figsize=(12,6))
sns.barplot(data=northeast_vacc_melted, x='State', y='value', hue='variable')
plt.title("Vaccination Rates in Northeast India - NFHS-4 vs NFHS_5")
plt.xlabel('State')
plt.ylabel('Full Vaccination Rate(%)')
plt.xticks(rotation=45)
plt.legend(title="Survey Round")
plt.savefig('../outputs/northeast_vacc_comparison.png', bbox_inches='tight')
plt.show()

The chart above compares full vaccination rates across the 8 different states of the northeast between NFHS-4(2015-2016) & NFHS-5(2019-2021).

Key findings
- Arunachal Pradesh and Tripura made the most significant improvements, from under 20% to over 60%
- Sikkim remains the highest performer, with above 80% for both NFHS-4 and NFHS-5
- Manipur showed the smallest improvement, with no major change for both surveys
- Nagaland remains the lowest performer, under 40% in NFHS-4 and 59% in NFHS-5

All eight states showed improvement, bucking concerns about regional stagnation — however Manipur and Nagaland remain significantly below the national average of 76.8%.

In [ ]:
df[df['Category'] == 'Maternal and Child Health']['Indicator'].unique()

In [21]:
matr_df = df[df['Indicator'] == 'Mothers who had at least 4 antenatal care visits (%)']

In [ ]:
matr_df.shape

In [ ]:
matr_df[matr_df["NFHS_5"] < 5]

In [24]:
matr_clean = matr_df[matr_df['State'] != 'Jammu & Kashmir']

In [ ]:
zscores = stats.zscore(matr_clean['NFHS_5'])
matr_clean[abs(zscores) > 3]

Zscore outlier check found no districts beyond 3 standard deviations 
for antenatal care visits, suggesting more uniform distribution 
compared to vaccination rates.

In [26]:
northeast_matr = matr_clean[matr_clean['State'].isin(['Assam', 'Manipur', 'Nagaland', 'Meghalaya', 'Mizoram', 'Tripura', 'Arunachal Pradesh', 'Sikkim'])]

In [ ]:
northeast_matr.shape

In [28]:
northeast_avg = northeast_matr.groupby('State')[['NFHS_5', 'NFHS_4']].mean()

In [29]:
northeast_avg = northeast_avg.reset_index()

In [ ]:
northeast_avg.columns

In [31]:
northeast_matr_melted = pd.melt(northeast_avg, id_vars = ['State'], value_vars = ['NFHS_5', 'NFHS_4'])

In [ ]:
plt.figure(figsize=(12,6))
sns.barplot(data=northeast_matr_melted, x='State', y='value', hue='variable')
plt.title("Antenatal Care Visits in Northeast India - NFHS-4 vs NFHS-5")
plt.xlabel('State')
plt.ylabel('Mothers with 4+ Antenatal Visits (%)')
plt.xticks(rotation=45)
plt.legend(title="Survey Round")
plt.savefig('../outputs/northeast_matr_comparison.png', bbox_inches='tight')
plt.show()

The chart compares antenatal care visits in Northeast India between NFHS-4(2015-16) & NFHS-5(2019-2021).

Key findings
- Sikkim declined from 78% in NFHS-4 to 65% in NFHS-5
- Tripura, Meghalaya, Assam and Arunachal made major improvements, less than 15% in NFHS-4 to 35-45% in NFHS-5
- Manipur remains the highest in NFHS-5 at 70% 
- Mizoram declined slightly from 56% to 55%
- Nagaland remains the lowest, below 20% in both surveys

Notably, Manipur leads the Northeast in antenatal care despite 
performing poorly on vaccination rates, suggesting maternal 
healthcare infrastructure is stronger than childhood immunization delivery.

In [ ]:
df[df['Category'] == 'Child Feeding Practices and Nutritional Status of Children']['Indicator'].unique() 

In [34]:
mal_df = df[df['Indicator'] == 'Children under 5 years who are stunted (height for age) (%)']

In [ ]:
mal_df.shape

In [ ]:
mal_df[mal_df['NFHS_5'] < 5]

In [37]:
mal_clean = mal_df[mal_df['State'] != 'Jammu & Kashmir']

In [ ]:
zscores = stats.zscore(mal_clean['NFHS_5'])
mal_clean[abs(zscores) > 3]

Data suggests Jharkhand is India's most malnutritionally challenges states; 60.6% children under 5 are stunted

In [39]:
northeast_mal = mal_clean[mal_clean['State'].isin(['Assam', 'Manipur', 'Nagaland', 'Meghalaya', 'Mizoram', 'Tripura', 'Arunachal Pradesh', 'Sikkim'])]

In [ ]:
northeast_mal.shape

In [41]:
northeast_av = northeast_mal.groupby('State')[['NFHS_4', 'NFHS_5']].mean()

In [42]:
northeast_av = northeast_av.reset_index()

In [ ]:
northeast_av.columns

In [44]:
northeast_mal_melted = pd.melt(northeast_av, id_vars = ['State'], value_vars = ['NFHS_5', 'NFHS_4'])

In [ ]:
plt.figure(figsize=(12,6))
sns.barplot(data=northeast_mal_melted, x='State', y='value', hue='variable')
plt.title("Children under 5 years who are stunted - NFHS-4 vs NFHS-5")
plt.xlabel('State')
plt.ylabel('Children under 5 stunted (%)')
plt.xticks(rotation=45)
plt.legend(title="Survey Round")
plt.savefig('../outputs/northeast_mal_comparison.png', bbox_inches='tight')
plt.show()

The chart above shows the percentage of children under 5 years who are 
stunted (height for age) across eight Northeast states between 
NFHS-4 (2015-16) and NFHS-5 (2019-21).

Key findings:
- Every Northeast state except Manipur and Sikkim saw stunting rates worsen
- Meghalaya showed the most alarming increase, jumping from 10.7% to 43.3%
- Tripura nearly doubled from 3.9% to 33.2%
- Manipur was the only major improver, declining from 31.1% to 24.6%
- Sikkim also showed slight improvement, from 31.1% to 25%

Context:
The worsening across most states likely reflects disruption to 
nutrition programs like ICDS during the COVID-19 pandemic, which 
overlapped with the NFHS-5 data collection period (2019-21).
Manipur's improvement despite regional trends suggests stronger 
nutrition program delivery at the district level.

In [ ]:
vacc_clean[vacc_clean['State'] == 'Manipur']['NFHS_5'].mean()

In [ ]:
vacc_clean['NFHS_5'].mean()

In [ ]:
matr_clean[matr_clean['State'] == 'Manipur']['NFHS_5'].mean()

In [ ]:
matr_clean['NFHS_5'].mean()

In [ ]:
mal_clean[mal_clean['State'] == 'Manipur']['NFHS_5'].mean()

In [ ]:
mal_clean['NFHS_5'].mean()

Manipur built strong facility based maternal care but lost the community outreach infrastructure that drove vaccination.

In [ ]:
categories = ['Manipur', 'National Average']
values = [64.8, 77.7]
plt.bar(categories, values, color = ['Green', 'Orange'])
plt.title("Vaccination rates - Manipur vs National average (NFHS-5)")
plt.ylabel("Vaccinated children (12-23 months) %")
plt.savefig('../outputs/manipur_vacc_comp.png', bbox_inches="tight")


In [ ]:
categories = ['Manipur', 'National Average']
values = [70.6, 59.5]
plt.bar(categories, values, color = ['Green', 'Orange'])
plt.title("Maternal health - Manipur vs National average (NFHS-5)")
plt.ylabel("Mothers with 4+ antenatal visit %")
plt.savefig('../outputs/manipur_matr_comp.png', bbox_inches="tight")

In [ ]:
categories = ['Manipur', 'National Average']
values = [24.6, 33.7]
plt.bar(categories, values, color = ['Green', 'Orange'])
plt.title("Child malnutrition- Manipur: National average (Lower = Better)")
plt.ylabel("Children under 5 stunted %")
plt.savefig('../outputs/manipur_mal_comp.png', bbox_inches="tight")

In [ ]:
vacc_clean.nlargest(n=10, columns = 'NFHS_5')

In [ ]:
vacc_clean.nsmallest(n=10, columns = 'NFHS_5')

In [ ]:
matr_clean.nlargest(n=10, columns = 'NFHS_5')

In [ ]:
matr_clean.nsmallest(n=10, columns = 'NFHS_5')

In [ ]:
mal_clean.nlargest(n=10, columns = 'NFHS_5')

In [ ]:
mal_clean.nsmallest(n=10, columns = 'NFHS_5')

In [61]:
mal_state_avg = mal_clean.groupby('State')['NFHS_5'].mean()

In [ ]:
mal_state_avg.sort_values(ascending=False).head(10)

In [ ]:
sorted_mal = mal_state_avg.sort_values(ascending=False)
values = sorted_mal.values
labels = sorted_mal.index
plt.figure(figsize=(10,15))
plt.plot(values, labels, 'o')
plt.savefig('../outputs/dot_chart_malnutrtion.png', bbox_inches="tight")

In [90]:
vacc_merge = vacc_clean[['NFHS_5', 'District', 'DT_CEN_CD', 'State']].rename(columns={'NFHS_5': 'vaccination rates'})

In [91]:
matr_merge = matr_clean[['NFHS_5', 'District', 'DT_CEN_CD', 'State']].rename(columns={'NFHS_5': 'antenatal care'})

In [92]:
mal_merge = mal_clean[['NFHS_5', 'District', 'DT_CEN_CD', 'State']].rename(columns={'NFHS_5': 'child malnutrition'})

In [93]:
temp_df = pd.merge(vacc_merge, matr_merge, on=['State','District'], how='inner')

In [95]:
health_df = pd.merge(temp_df, mal_merge, on=['State','District'], how='inner')

In [99]:
health_df = health_df.drop(columns=['DT_CEN_CD_x', 'DT_CEN_CD_y', 'DT_CEN_CD'])